In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("vendas_loja.csv")

In [ ]:
df["receita"] = df["preco"] * df["quantidade"]
df.head() #head sem parametro puxa os 5 primeiros

,id_venda,data,produto,categoria,preco,quantidade,cidade,pagamento,receita
0,1,2026-10-01,Camiseta,Roupa,50,3,Teresina,Pix,150
1,2,2026-10-01,Tênis,Calçado,250,1,Recife,Cartão,250
2,3,2026-10-02,Boné,Acessório,40,2,Fortaleza,Pix,80
3,4,2026-10-02,Calça,Roupa,120,2,São Paulo,Cartão,240
4,5,2026-10-03,Mochila,Acessório,150,1,Teresina,Dinheiro,150


In [5]:
df.groupby("categoria")["receita"].sum() # o groupby separa em grupos para fazer alterações nele observando algum fator

categoria
Acessório     970
Calçado      1450
Roupa        2120
Name: receita, dtype: int64

In [6]:
df.groupby("cidade")["receita"].mean() # vai aparecer a media das receitas por cidade

cidade
Fortaleza    192.000000
Recife       186.000000
São Paulo    282.500000
Teresina     253.333333
Name: receita, dtype: float64

In [ ]:
resumo = df.groupby("categoria").agg(
    receita_total=("receita", "sum"),
    receita_media=("receita", "mean"),
    n_vendas=("id_venda", "count")
)
resumo # o .agg é usado para varias metricas ao mesmo tempo

,receita_total,receita_media,n_vendas
categoria,,,
Acessório,970,138.571429,7
Calçado,1450,290.000000,5
Roupa,2120,265.000000,8


In [ ]:
df.groupby("cidade")["receita"].sum().sort_values(ascending=False) # do maior para o menor

cidade
Teresina     1520
São Paulo    1130
Fortaleza     960
Recife        930
Name: receita, dtype: int64

In [11]:
""" 
pivot_table é a tabela dinâmica do Excel, dentro do Python 
values: o que será calculado.
index: o que vira linha.
columns: o que vira coluna.
aggfunc: a conta (soma, média, contagem).
fill_value=0: coloca zero onde não há vendas.
 """

' \npivot_table é a tabela dinâmica do Excel, dentro do Python \nvalues: o que será calculado.\nindex: o que vira linha.\ncolumns: o que vira coluna.\naggfunc: a conta (soma, média, contagem).\nfill_value=0: coloca zero onde não há vendas.\n '

In [10]:
pd.pivot_table(
    df,
    values="receita",
    index="cidade",
    columns="categoria",
    aggfunc="sum",
    fill_value=0,
)

categoria,Acessório,Calçado,Roupa
cidade,,,
Fortaleza,530,430,0
Recife,40,520,370
São Paulo,90,500,540
Teresina,310,0,1210


In [ ]:
"""Quando as duas tabelas têm uma coluna em comum,
o merge consegue juntá-las. A coluna em comum se chama chave."""

In [14]:
catalogo = pd.DataFrame({
    "produto": ["Camiseta", "Tênis", "Boné", "Calça", "Mochila", "Sandália", "Jaqueta", "Meia", "Cinto"],
    "fornecedor": ["Malharia Sul", "Passo Firme", "Malharia Sul", "Jeans Norte", "Bolsas Brasil",
                   "Passo Firme", "Jeans Norte", "Malharia Sul", "Couro Forte"],
    "custo_unitario": [25, 130, 15, 60, 70, 40, 150, 5, 20],
})
catalogo

,produto,fornecedor,custo_unitario
0,Camiseta,Malharia Sul,25
1,Tênis,Passo Firme,130
2,Boné,Malharia Sul,15
3,Calça,Jeans Norte,60
4,Mochila,Bolsas Brasil,70
5,Sandália,Passo Firme,40
6,Jaqueta,Jeans Norte,150
7,Meia,Malharia Sul,5
8,Cinto,Couro Forte,20


In [ ]:
df2 = df.merge(catalogo, on="produto", how="left")
df2.head() #juntou a tabela vendas com a tabela catalogo que tem produto como chave entre elas

,id_venda,data,produto,categoria,preco,quantidade,cidade,pagamento,receita,fornecedor,custo_unitario
0,1,2026-10-01,Camiseta,Roupa,50,3,Teresina,Pix,150,Malharia Sul,25
1,2,2026-10-01,Tênis,Calçado,250,1,Recife,Cartão,250,Passo Firme,130
2,3,2026-10-02,Boné,Acessório,40,2,Fortaleza,Pix,80,Malharia Sul,15
3,4,2026-10-02,Calça,Roupa,120,2,São Paulo,Cartão,240,Jeans Norte,60
4,5,2026-10-03,Mochila,Acessório,150,1,Teresina,Dinheiro,150,Bolsas Brasil,70


In [16]:
df.shape

(20, 9)

In [ ]:
df2.shape # aumentou duas colunas

(20, 11)

In [ ]:
"""O how responde: “o que fazer quando um produto existe em uma tabela mas não na outra?”.
Pense em duas figuras sobrepostas, a de vendas (esquerda) e a de catálogo (direita):

how=	O que mantém
"left"	Todas as linhas da tabela da esquerda (df), mesmo sem par no catálogo
"inner"	Só as linhas que existem nas duas
"outer"	Todas as linhas das duas tabelas"""

In [18]:
df2["lucro"] = (df2["preco"] - df2["custo_unitario"]) * df2["quantidade"]

In [19]:
df2[["produto", "preco", "custo_unitario", "quantidade", "lucro"]].head()

,produto,preco,custo_unitario,quantidade,lucro
0,Camiseta,50,25,3,75
1,Tênis,250,130,1,120
2,Boné,40,15,2,50
3,Calça,120,60,2,120
4,Mochila,150,70,1,80


In [21]:
df2.groupby("fornecedor")["lucro"].sum().sort_values(ascending=False)

fornecedor
Jeans Norte      810
Passo Firme      730
Malharia Sul     585
Bolsas Brasil    240
Name: lucro, dtype: int64

In [23]:
primeira = df.iloc[:10]
segunda = df.iloc[10:]
tudo = pd.concat([primeira, segunda], ignore_index=True)
# concat coloca os valores de uma tabela embaixo da outra, é usado para o mesmo tipo de coluna mas em arquivos diferentes
tudo.shape

(20, 9)

In [27]:
df.groupby("categoria")["receita"].sum().sort_values(ascending=False)

categoria
Roupa        2120
Calçado      1450
Acessório     970
Name: receita, dtype: int64

In [28]:
df.groupby("cidade")["receita"].mean().sort_values(ascending=False)

cidade
São Paulo    282.500000
Teresina     253.333333
Fortaleza    192.000000
Recife       186.000000
Name: receita, dtype: float64

In [29]:
df.groupby("pagamento")["quantidade"].sum()

pagamento
Cartão      13
Dinheiro     4
Pix         37
Name: quantidade, dtype: int64

In [35]:
df2.groupby("fornecedor")["lucro"].sum().sort_values(ascending=False)

fornecedor
Jeans Norte      810
Passo Firme      730
Malharia Sul     585
Bolsas Brasil    240
Name: lucro, dtype: int64

In [36]:
tabela = pd.pivot_table(
    df,
    values="receita",
    index="pagamento",
    columns="categoria",
    aggfunc="sum",
    fill_value=0,
)
tabela

categoria,Acessório,Calçado,Roupa
pagamento,,,
Cartão,300,430,1000
Dinheiro,190,250,120
Pix,480,770,1000
